In [1]:
import requests
import xml.etree.ElementTree as ET

In [2]:
res = requests.get("https://g1.globo.com/sitemap/g1/2026/04/13_1.xml")
root = ET.fromstring(res.content)
root

<Element '{http://www.sitemaps.org/schemas/sitemap/0.9}urlset' at 0x700c1e9dfba0>

In [3]:
root.tag

'{http://www.sitemaps.org/schemas/sitemap/0.9}urlset'

In [4]:
import xmltodict

In [5]:
data = xmltodict.parse(res.content)
data


{'urlset': {'@xmlns': 'http://www.sitemaps.org/schemas/sitemap/0.9',
  '@xmlns:xhtml': 'http://www.w3.org/1999/xhtml',
  '@xmlns:image': 'http://www.google.com/schemas/sitemap-image/1.1',
  'url': [{'loc': 'https://g1.globo.com/sp/sorocaba-jundiai/noticia/2026/04/13/adolescente-morre-por-h1n1-em-sorocaba.ghtml',
    'lastmod': '2026-04-14T17:30:15.338000+00:00',
    'image:image': {'image:loc': 'https://s2-g1.glbimg.com/5WMQUedvi2zqpNQj2e_POdc2RCs=/i.s3.glbimg.com/v1/AUTH_59edd422c0c84a879bd37670ae4f538a/internal_photos/bs/2023/T/d/RKq3A5R1uhACEssrjRpg/vacina-sorocaba.jpg'}},
   {'loc': 'https://g1.globo.com/sp/vale-do-paraiba-regiao/noticia/2026/04/13/exportacoes-recuam-2percent-no-primeiro-trimestre-de-2026-no-vale-do-paraiba-e-regiao-bragantina.ghtml',
    'lastmod': '2026-04-13T23:56:07.685000+00:00',
    'image:image': {'image:loc': 'https://s2-g1.glbimg.com/G1v13EIEXgZUaPQyp-3jJBT-4Es=/i.s3.glbimg.com/v1/AUTH_59edd422c0c84a879bd37670ae4f538a/internal_photos/bs/2026/A/0/P7oAY3TpGa

In [6]:
locs = map(lambda x: x["loc"], data["urlset"]["url"])
locs

In [7]:
titles = list(map(lambda x: " ".join(x.split("/")[-1].split("-")[:-1]), locs))

In [8]:
print(len(titles))
for title in titles:
    print(title)

752
adolescente morre por h1n1 em
exportacoes recuam 2percent no primeiro trimestre de 2026 no vale do paraiba e regiao
trump chama leao xiv de fraco e pessimo papa reage e diz que vai seguir firme contra a
marilia sanciona lei que permite enterro de caes e gatos em jazigos de tutores projeto e aprovado na camara em
video onibus e destruido por incendio na br 364 em
medico e detido acusado de agredir companheira no interior de
apos pressao do governo projeto que regulamenta atividade de entregadores e motoristas sai de pauta na
zoologico de fortaleza vai fechar durante dois dias apos lotacoes registradas desde a
medico moto atingida
moradores reclamam de vizinho que acumula lixo e entulho em casa no acre extremamente
corregedoria do tribunal de justica do amazonas analisa omissao de juiz do careiro apos demora em decisao diz
pm da reserva tentar matar esposa a tiros e vitima escapa ao pular muro de casa e pedir socorro em bar de
apos dois anos policia federal ainda tenta identificar co

In [9]:
import spacy

/home/rafael-albuquerque/repos/tcc/.venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [10]:
nlp = spacy.load("pt_core_news_sm")

In [11]:
import pandas as pd

In [12]:
x = nlp(titles[0])

print(x.ents)
for token in x:
    print(token.text, token.lemma_, token.pos_, token.tag_, token.dep_,
          token.shape_, token.is_alpha, token.is_stop)

(h1n1,)
adolescente adolescente ADV ADV advmod xxxx True False
morre morrer VERB VERB ROOT xxxx True False
por por ADP ADP case xxx True True
h1n1 h1n1 NOUN NOUN obl xdxd False False
em em ADP ADP dep xx True True


In [13]:
subset = titles[:20]

In [14]:
df = pd.DataFrame(subset, columns=["title"])
df["entities"] = df["title"].apply(lambda x: nlp(x).ents)
df

,title,entities
0,adolescente morre por h1n1 em,"((h1n1),)"
1,exportacoes recuam 2percent no primeiro trimes...,"((exportacoes),)"
2,trump chama leao xiv de fraco e pessimo papa r...,"((trump, chama, leao, xiv, de, fraco),)"
3,marilia sanciona lei que permite enterro de ca...,"((marilia), (camara))"
4,video onibus e destruido por incendio na br 36...,"((br, 364),)"
5,medico e detido acusado de agredir companheira...,()
6,apos pressao do governo projeto que regulament...,()
7,zoologico de fortaleza vai fechar durante dois...,()
8,medico moto atingida,()
9,moradores reclamam de vizinho que acumula lixo...,"((acre),)"


In [15]:
import ollama

In [16]:
def rewrite_title(title):
    client = ollama.Client("http://192.168.0.12:11434")
    prompt = "Reescreva o título da notícia com base nessa versão extraída de uma URL. Esse texto não tem capitalização e nem pontuação, e é sua responsabilidade adicionar capitalização e pontuação onde for adequado.\nTexto original: {}\nTexto reescrito: ".format(title)
    response = client.generate("llama3.2:3b", prompt).response
    return response

In [21]:
df["rewritten_title"] = df["title"].apply(rewrite_title)
df

,title,entities,rewritten_title
0,adolescente morre por h1n1 em,"((h1n1),)",Adolescente Morre Por H1N1 Em
1,exportacoes recuam 2percent no primeiro trimes...,"((exportacoes),)","""Exportações Recuam 2% no Primeiro Trimestre d..."
2,trump chama leao xiv de fraco e pessimo papa r...,"((trump, chama, leao, xiv, de, fraco),)","Trump chama Léonidas XIV de fraco e péssimo, P..."
3,marilia sanciona lei que permite enterro de ca...,"((marilia), (camara))","""Marília Sanciona Lei que Permite Enterro de C..."
4,video onibus e destruido por incendio na br 36...,"((br, 364),)","""VIDEO: ONIBUS DESTRUIDO POR INCÊNDIO NA BR 364"""
5,medico e detido acusado de agredir companheira...,(),"""Ação contra médico que detido por acusado de ..."
6,apos pressao do governo projeto que regulament...,(),"""A Pauta do Governo: Projeto que Regula Ativid..."
7,zoologico de fortaleza vai fechar durante dois...,(),"""Zoológico de Fortaleza Fecha por Dois Dias Ap..."
8,medico moto atingida,(),Claro! Aqui está a versão reescrita do título ...
9,moradores reclamam de vizinho que acumula lixo...,"((acre),)","""Moradores Reclamam de Vizinho com Lixo e Entu..."


In [20]:
for i, title, rt in df[["title", "rewritten_title"]].itertuples():
    print(f"{i} | {title} | {rt}")

0 | adolescente morre por h1n1 em | "Adolescente morre por h1n1."
1 | exportacoes recuam 2percent no primeiro trimestre de 2026 no vale do paraiba e regiao | "Exportações Recuam 2% no Primeiro Trimestre de 2026, diz Estudo do Vale do Paraíba e Região."
2 | trump chama leao xiv de fraco e pessimo papa reage e diz que vai seguir firme contra a | "Trump chama Leão XIV de fraco e pobre, Papa - Reação do Papa Pio XI."
3 | marilia sanciona lei que permite enterro de caes e gatos em jazigos de tutores projeto e aprovado na camara em | Marilia Sanciona Lei que Permite Enterro de Cães e Gatos em Jazigos de Tutores, Projeto e Aprovado na Câmara.
4 | video onibus e destruido por incendio na br 364 em | "Vídeo: Ônibus destruído por incêndio na BR-364".
5 | medico e detido acusado de agredir companheira no interior de | "Medico Detido Acusado de Agredir Companheira no Interior"

Essa é uma versão reescrita do título com base na versão extraída da URL, adicionando capitalização e pontuação para torn